# Multi-tensor sample with mixed variable / fixed sizes

This example mimics a **fingerprint verification** dataset where each sample
carries **6 tensors**:

| Item | Shape           | Dtype   | Variable? | Meaning                        |
|------|-----------------|---------|-----------|--------------------------------|
| `x1` | `(N, 96, 96)`   | float32 | yes       | N fingerprint patches          |
| `x2` | `(N,)`          | float32 | yes       | per-patch quality              |
| `x3` | `(N,)`          | float32 | yes       | per-patch score                |
| `x4` | `(N,)`          | float32 | yes       | per-patch confidence           |
| `x5` | `(N,)`          | int64   | yes       | per-patch finger id            |
| `x6` | `(1,)`          | int64   | **no**    | class label (subject id)       |

`N` varies per sample. The first 5 tensors must be padded to a common
`MAX_N` so they can be batched; the label `x6` is a fixed scalar and
should be returned as-is (no length, no padding).

We use the new `is_variable=[...]` flag added to both `VVTKDataset` and
`VVTKDataLoader` to mark which items are variable-length.

In [ ]:
import os, shutil, numpy as np, torch
from torch.utils.data import DataLoader
from vvtk_dataset import VVTKDataset, VVTKDataLoader

DATA_DIR    = 'fingerprint_data'
NUM_SAMPLES = 2000
NUM_CLASSES = 50
MAX_N       = 16        # max patches per sample (padding target)
BATCH_SIZE  = 32

if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Generate synthetic data and write the dataset

Each sample has a random `N` in `[1, MAX_N]`. The 6 tensors are written
uncompressed (fastest for small tensors).

In [ ]:
rng = np.random.default_rng(0)

with VVTKDataset(
    os.path.join(DATA_DIR, 'train'),
    mode='wb',
    num_shards=8,
    compression=['none'] * 6,
) as ds:
    for i in range(NUM_SAMPLES):
        n = int(rng.integers(1, MAX_N + 1))
        x1 = rng.standard_normal((n, 96, 96), dtype=np.float32)
        x2 = rng.standard_normal(n, dtype=np.float32)
        x3 = rng.standard_normal(n, dtype=np.float32)
        x4 = rng.standard_normal(n, dtype=np.float32)
        x5 = rng.integers(0, 10, size=n, dtype=np.int64)
        x6 = np.array([i % NUM_CLASSES], dtype=np.int64)   # scalar label
        ds.add(i, x1, x2, x3, x4, x5, x6)

print(f'Wrote {NUM_SAMPLES} samples.')

## 2. Read with the standard `torch.utils.data.DataLoader`

We open the dataset with `fixed_shapes` + `is_variable` so that
`__getitem__` automatically pads the first 5 tensors to `MAX_N` and
returns `x6` as a plain tensor.

In [ ]:
ds = VVTKDataset(
    os.path.join(DATA_DIR, 'train'),
    mode='rb',
    compression=['none'] * 6,
    fixed_shapes=[(MAX_N, 96, 96), (MAX_N,), (MAX_N,), (MAX_N,), (MAX_N,), (1,)],
    padding_values=[0.0, 0.0, 0.0, 0.0, -1, 0],
    is_variable=[True, True, True, True, True, False],
)

sample = ds[0]
print('Number of items returned :', len(sample))
for i, item in enumerate(sample):
    if isinstance(item, tuple):
        t, length = item
        print(f'  x{i+1}: tensor {tuple(t.shape)} {t.dtype}   length={length.item()}')
    else:
        print(f'  x{i+1}: tensor {tuple(item.shape)} {item.dtype}   value={item.tolist()}')

In [ ]:
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

batch = next(iter(loader))
(x1, l1), (x2, l2), (x3, l3), (x4, l4), (x5, l5), x6 = batch
print('Batch shapes via torch DataLoader:')
print('  x1', x1.shape, x1.dtype, '  length', l1.shape)
print('  x2', x2.shape, x2.dtype, '  length', l2.shape)
print('  x5', x5.shape, x5.dtype, '  length', l5.shape)
print('  x6 (label)', x6.shape, x6.dtype)

## 3. Read with the C++ `VVTKDataLoader`

The same `is_variable=[...]` flag controls the output format: variable
items come back as `(data, length)` pairs, fixed items come back as a
plain tensor.

In [ ]:
ds_cpp = VVTKDataset(
    os.path.join(DATA_DIR, 'train'),
    mode='rb',
    compression=['none'] * 6,
)

loader_cpp = VVTKDataLoader(
    ds_cpp,
    batch_size=BATCH_SIZE,
    num_workers=4,
    ring_size=4,
    shapes=[(MAX_N, 96, 96), (MAX_N,), (MAX_N,), (MAX_N,), (MAX_N,), (1,)],
    dtypes=[torch.float32, torch.float32, torch.float32, torch.float32,
            torch.int64, torch.int64],
    padding_values=[0.0, 0.0, 0.0, 0.0, -1, 0],
    is_variable=[True, True, True, True, True, False],
    shuffle=True,
)

batch = next(iter(loader_cpp))
(x1, l1), (x2, l2), (x3, l3), (x4, l4), (x5, l5), x6 = batch
print('Batch shapes via VVTKDataLoader (C++):')
print('  x1', x1.shape, x1.dtype, '  length', l1.shape, 'sample lens =', l1[:5].tolist())
print('  x2', x2.shape, x2.dtype, '  length', l2.shape)
print('  x5', x5.shape, x5.dtype, '  length', l5.shape)
print('  x6 (label)', x6.shape, x6.dtype, '  first 5:', x6[:5].tolist())
print(f'  total batches in epoch: {len(loader_cpp)}')

## 4. Tiny end-to-end training step

A toy classifier that masks out padded patches using the lengths returned
by the loader, then averages the per-patch features and predicts `x6`.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, 3, padding=1)
        self.fc   = nn.Linear(8 * 96 * 96, num_classes)

    def forward(self, x1, lens):
        B, N, H, W = x1.shape
        f = F.relu(self.conv(x1.reshape(B * N, 1, H, W)))
        f = f.reshape(B, N, -1)
        mask = (torch.arange(N, device=f.device)[None, :] < lens[:, None]).float()
        pooled = (f * mask.unsqueeze(-1)).sum(1) / lens.clamp(min=1).unsqueeze(-1).float()
        return self.fc(pooled)

model = Net(NUM_CLASSES)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

for step, batch in enumerate(loader_cpp):
    (x1, l1), _, _, _, _, x6 = batch
    logits = model(x1, l1)
    loss   = F.cross_entropy(logits, x6)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0:
        print(f'step {step:3d}  loss={loss.item():.3f}')
    if step >= 30:
        break